# 15. Backward, Forward and Stepwise feature selection

Three predictive selection strategies, identical candidate pool and evaluation conditions:

| Method | Start | Each step |
|---|---|---|
| Backward | All 34 candidate columns | Best individual numeric deletion |
| Forward | Weekday and ZIP fixed effects | Best individual numeric addition |
| Stepwise | Weekday and ZIP fixed effects | Best addition, followed by repeated deletion checks |

Month is a single categorical unit; weekday controls and ZIP FE remain fixed. Every accepted change requires a fresh refit. The shared stopping threshold is a **0.1% relative improvement in mean selection MAE**, a declared heuristic, not a statistical significance rule. P-values describe paired forecast-loss contrasts and do not determine membership.

Use the same 585-region cohort, 30-day rolling training windows, weekly refits, and direct forecasts for days 1/2/3. Select using January-June 2021, then freeze each list before scoring July-December 2021. Later observations can enter subsequent rolling parameter refits. Earlier work examined 2021, so later validation is not a historically untouched holdout. No 2022-2023 test evaluation. Differences across these searches are method-dependent candidates, not a universal true feature set.

Exact duplicate feature representations and static FE-absorbed attributes are handled as in Notebook 14. Missing population regions are excluded consistently from every method; missing rates are never treated as observed zeros. Self-tests verify cached subset fitting against direct within OLS.


In [3]:
"""Validation-only feature ablations and exploratory inference for NJ onset counts.

Requires numpy/pandas. Optional matplotlib creates report figures. No GMM tests.
"""
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import math
import platform
import numpy as np
import pandas as pd

REPO=Path(r'C:\Users\41240\OneDrive\文档\GitHub\practicum-project\Weikun')
SOURCE=REPO/'nj_daily_feature_outputs'/'nj_daily_features.csv.gz'

BASE=['cases_today','cases_lag_1','cases_lag_7']
HISTORY=BASE+['cases_mean_7d']
EXTENDED=HISTORY+['cases_lag_14','case_std_7d']
ACC=EXTENDED+['case_acceleration']
DYNAMICS=ACC+['case_jerk']
SPECIFICATIONS={
    'B0_basic':BASE,
    'H0_weekly_mean':HISTORY,
    'H1_history_extended':EXTENDED,
    'D1_acceleration':ACC,
    'D2_jerk':DYNAMICS,
    'P0_basic_pressure7':BASE+['nj_covid_pressure_7d'],
    'P1_rich_pressure7':DYNAMICS+['nj_covid_pressure_7d'],
    'P2_rich_pressure14':DYNAMICS+['nj_covid_pressure_14d'],
}
COMPARISONS=[
    ('B0_basic','H0_weekly_mean','weekly_mean'),
    ('H0_weekly_mean','H1_history_extended','lag14_and_volatility'),
    ('H1_history_extended','D1_acceleration','acceleration'),
    ('D1_acceleration','D2_jerk','jerk'),
    ('B0_basic','P0_basic_pressure7','pressure7_basic'),
    ('D2_jerk','P1_rich_pressure7','pressure7_rich'),
    ('D2_jerk','P2_rich_pressure14','pressure14_rich'),
    ('P1_rich_pressure7','P2_rich_pressure14','pressure14_vs_pressure7'),
]


def two_way_meat(scores,n_parameters):
    """CR1 two-way cluster sandwich meat: region + date - region/date cell."""
    N,T,p=scores.shape
    n=N*T
    if min(N,T)<=1 or n<=n_parameters:
        return np.full((p,p),np.nan)
    base=(n-1)/(n-n_parameters)
    by_region=scores.sum(axis=1)
    by_date=scores.sum(axis=0)
    cells=scores.reshape(-1,p)
    return base*(N/(N-1)*(by_region.T@by_region)+T/(T-1)*(by_date.T@by_date)-n/(n-1)*(cells.T@cells))


def fit_fe(x,y):
    N,T,p=x.shape
    assert np.isfinite(x).all() and np.isfinite(y).all()
    xm=x.mean(axis=1); ym=y.mean(axis=1)
    xc=x-xm[:,None,:]; yc=y-ym[:,None]
    scale=np.sqrt(np.mean(xc*xc,axis=(0,1)))
    scale=np.where(scale>1e-12,scale,1)
    z=xc/scale
    zf=z.reshape(-1,p)
    b,_,rank,sv=np.linalg.lstsq(zf,yc.reshape(-1),rcond=None)
    beta=b/scale; alpha=ym-xm@beta
    residual=yc-np.einsum('ntp,p->nt',z,b)
    if rank==p:
        bread=np.linalg.pinv(zf.T@zf,rcond=1e-12)
        covariance=bread@two_way_meat(z*residual[:,:,None],N+p)@bread
        variance=np.diag(covariance)/(scale*scale)
        se=np.full(p,np.nan)
        positive=np.isfinite(variance)&(variance>0)
        se[positive]=np.sqrt(variance[positive])
    else:
        se=np.full(p,np.nan)
    pvalue=np.array([math.erfc(abs(v/s)/math.sqrt(2)) if np.isfinite(s) and s>0 else np.nan for v,s in zip(beta,se)])
    condition=float(sv[0]/sv[-1]) if len(sv) and sv[-1]>0 else np.inf
    return beta,alpha,se,pvalue,int(rank),condition


def hac_mean_test(d,bandwidth=14):
    """Two-sided asymptotic test of zero mean paired DATE-level loss difference."""
    d=np.asarray(d,float)
    assert np.isfinite(d).all()
    T=len(d); L=min(int(bandwidth),T-1)
    mean=float(d.mean()); c=d-mean
    longvar=float(c@c/T)
    for lag in range(1,L+1):
        longvar+=2*(1-lag/(L+1))*float(c[lag:]@c[:-lag]/T)
    variance=longvar/T*T/(T-1)
    if variance<=0 or not np.isfinite(variance):
        return mean,np.nan,np.nan
    se=math.sqrt(variance)
    return mean,se,math.erfc(abs(mean/se)/math.sqrt(2))


def holm_adjust(p):
    p=np.asarray(p,float); adjusted=np.full(p.shape,np.nan)
    good=np.flatnonzero(np.isfinite(p)); ordered=good[np.argsort(p[good])]
    if len(ordered):
        values=np.maximum.accumulate((len(ordered)-np.arange(len(ordered)))*p[ordered])
        adjusted[ordered]=np.minimum(values,1)
    return adjusted


def block_indices(T,block_length,repetitions,seed):
    """Noncircular moving-block resampling; one date index shared by all regions/horizons."""
    assert 1<=block_length<=T
    rng=np.random.default_rng(seed)
    starts=rng.integers(0,T-block_length+1,size=(repetitions,math.ceil(T/block_length)))
    return (starts[:,:,None]+np.arange(block_length)[None,None,:]).reshape(repetitions,-1)[:,:T]


def residual_correlation(errors,lag):
    a=errors[:,lag:]; b=errors[:,:-lag]
    a=a-a.mean(axis=1,keepdims=True); b=b-b.mean(axis=1,keepdims=True)
    den=np.sqrt(np.sum(a*a,axis=1)*np.sum(b*b,axis=1))
    value=np.full(len(a),np.nan)
    good=den>0; value[good]=np.sum(a[good]*b[good],axis=1)/den[good]
    return value


def self_checks():
    # FWL estimates and cluster covariance versus an explicit dummy design.
    rng=np.random.default_rng(729)
    N,T,p=5,12,2
    x=rng.normal(size=(N,T,p)); y=x@np.array([.5,-.25])+np.arange(N)[:,None]+rng.normal(size=(N,T))
    beta,alpha,se,pv,rank,_=fit_fe(x,y)
    design=np.column_stack([np.repeat(np.eye(N),T,axis=0),x.reshape(-1,p)])
    full=np.linalg.lstsq(design,y.reshape(-1),rcond=None)[0]
    assert np.allclose(full[:N],alpha) and np.allclose(full[N:],beta)
    e=y.reshape(-1)-design@full
    inv=np.linalg.pinv(design.T@design)
    covariance=inv@two_way_meat((design*e[:,None]).reshape(N,T,N+p),N+p)@inv
    positive=np.diag(covariance)[N:]>0
    assert np.allclose(se[positive]**2,np.diag(covariance)[N:][positive])
    assert np.allclose(holm_adjust([.01,.04,.03]),[.03,.06,.06])
    independent=rng.normal(size=100)
    _,s,_=hac_mean_test(independent,0)
    assert np.isclose(s,np.std(independent,ddof=1)/10)



GROUPS = {
    'core_history': ['cases_today','cases_lag_1','cases_lag_7','cases_lag_14'],
    'rolling_level': ['cases_mean_7d','cases_mean_14d'],
    'volatility': ['case_std_7d','case_std_14d'],
    'dynamics': ['case_acceleration','case_jerk'],
    'local_rates': ['case_rate_today_per_100k','case_rate_7d_per_100k','case_rate_14d_per_100k'],
    'sparse_history': ['days_since_last_observed_case_censored','no_previous_observed_case'],
    'pressure': ['nj_covid_pressure_7d','nj_covid_pressure_14d'],
    'month': [f'target_month_{k}' for k in range(2,13)],
}
WEEKDAY=[f'target_weekday_{k}' for k in range(2,8)]

def independent_columns(x):
    # Training-only rank filtering. Removes algebraic duplicates and calendar
    # categories absent from the current window; not predictive feature selection.
    z=x-x.mean(axis=1,keepdims=True)
    norms=np.sqrt(np.mean(z*z,axis=(0,1)))
    selected=[]; q=[]
    for j in range(z.shape[2]):
        if norms[j]<1e-10: continue
        v=(z[:,:,j]/norms[j]).reshape(-1).copy()
        for _ in range(2):
            for u in q:v-=u*(u@v)
        length=np.linalg.norm(v)
        if length>1e-7*np.sqrt(v.size):
            selected.append(j);q.append(v/length)
    return selected


def run_three_method_selection(source,output,window=30,min_relative_improvement=0.001,bootstrap_repetitions=2000):
    """Greedy backward selection on early 2021; later 2021 checked after freezing."""
    self_checks()
    source=Path(source)
    folder=Path(output)/('three_methods_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
    folder.mkdir(parents=True)
    numeric=sum([v for k,v in GROUPS.items() if k!='month'],[])
    full=numeric+GROUPS['month']+WEEKDAY
    cols=sorted(set(numeric)-set(GROUPS['sparse_history']))
    p=pd.read_csv(source,usecols=cols+['zip','date','population','days_since_last_observed_case','pressure_definition'],dtype={'zip':'string'},parse_dates=['date'])
    assert p.pressure_definition.eq('valid_population_nj_origins_renormalized_v2').all()
    assert not p.duplicated(['zip','date']).any()
    ids=np.array(sorted(p.zip.unique()));dates=pd.date_range(p.date.min(),p.date.max())
    def matrix(c):return p.pivot(index='zip',columns='date',values=c).reindex(index=ids,columns=dates).to_numpy(float)
    population=matrix('population')[:,0]
    eligible=(population>0)&np.isfinite(population)&np.isfinite(matrix('nj_covid_pressure_14d')[:,13])
    matrices={c:matrix(c)[eligible] for c in cols}
    gap=matrix('days_since_last_observed_case')[eligible]
    matrices['no_previous_observed_case']=(~np.isfinite(gap)).astype(float)
    matrices['days_since_last_observed_case_censored']=np.where(np.isfinite(gap),gap,np.arange(len(dates))[None,:]+1)
    ids=ids[eligible];n=len(ids);counts=matrices['cases_today']
    origins=np.flatnonzero((dates>=pd.Timestamp('2021-01-01'))&((dates+pd.Timedelta(days=3))<=pd.Timestamp('2021-12-31')))
    v=len(origins);actual=np.stack([counts[:,origins+h] for h in (1,2,3)])
    selection_mask=(dates[origins]+pd.Timedelta(days=3))<=pd.Timestamp('2021-06-30')
    assessment_mask=dates[origins]>=pd.Timestamp('2021-07-01')
    assert not (selection_mask&assessment_mask).any()
    # Cached sufficient statistics let every candidate refit the actual OLS
    # on its own feature subset without repeating full raw training-data scans.
    cache=[];timing=[]
    for h in (1,2,3):
        target=dates+pd.Timedelta(days=h)
        for k in range(2,8):matrices[f'target_weekday_{k}']=np.broadcast_to((target.dayofweek==k-1).astype(float),(n,len(dates)))
        for k in range(2,13):matrices[f'target_month_{k}']=np.broadcast_to((target.month==k).astype(float),(n,len(dates)))
        x=np.stack([matrices[c] for c in full],axis=2)
        blocks=[]
        for first in range(0,v,7):
            block=origins[first:first+7];fit=int(block[0]);last=fit-h;train=np.arange(last-window+1,last+1)
            assert train[0]>=14 and train[-1]+h<=fit
            xt=x[:,train];yt=counts[:,train+h]
            assert np.isfinite(xt).all() and np.isfinite(x[:,block]).all()
            xm=xt.mean(axis=1);ym=yt.mean(axis=1);xc=xt-xm[:,None];yc=yt-ym[:,None]
            scale=np.sqrt((xc*xc).mean(axis=(0,1)));scale=np.where(scale>1e-10,scale,1)
            z=xc/scale;zf=z.reshape(-1,len(full))
            gram=zf.T@zf;rhs=zf.T@yc.reshape(-1)
            blocks.append((first,len(block),gram,rhs,scale,xm,ym,x[:,block]))
            timing.append({'horizon':h,'fit_date':dates[fit],'training_origin_start':dates[train[0]],'training_origin_end':dates[train[-1]],'latest_training_target':dates[train[-1]+h]})
        cache.append(blocks)
    np.seterr(invalid='raise')
    def predict(features,return_rank_log=False):
        ix=[full.index(f) for f in features]
        predictions=np.empty((3,n,v));rank_log=[]
        for hh,blocks in enumerate(cache):
            for first,length,g,rhs,scale,xm,ym,xp in blocks:
                # Ordered Gram-Schmidt in training Gram space removes only
                # nonidentified columns. Standardization is training-only.
                selected=[]
                for j in ix:
                    if g[j,j]<1e-8:continue
                    if not selected:residual=g[j,j]
                    else:
                        gg=g[np.ix_(selected,selected)];cross=g[selected,j]
                        residual=g[j,j]-cross@np.linalg.solve(gg,cross)
                    if residual>1e-8*max(g[j,j],1):selected.append(j)
                if selected:
                    b=np.linalg.solve(g[np.ix_(selected,selected)],rhs[selected])/scale[selected]
                    a=ym-xm[:,selected]@b
                    predictions[hh,:,first:first+length]=np.einsum('ntp,p->nt',xp[:,:,selected],b)+a[:,None]
                else:predictions[hh,:,first:first+length]=ym[:,None]
                if return_rank_log:rank_log.append({'horizon':hh+1,'forecast_origin':dates[origins[first]],'estimated_slopes':len(selected),'omitted':';'.join(full[j] for j in ix if j not in selected)})
        assert np.isfinite(predictions).all()
        return (predictions,rank_log) if return_rank_log else predictions
    def losses(pred,mask):
        e=np.maximum(pred,0)[:,:,mask]-actual[:,:,mask]
        return np.abs(e).mean(axis=1),(e*e).mean(axis=1),e.mean(axis=(1,2))
    # Numerical regression check: one subset must match direct within OLS.
    hh=0;first,length,g,rhs,scale,xm,ym,xp=cache[hh][0]
    check_features=['cases_today','cases_lag_1','cases_lag_7']+WEEKDAY
    check_ix=[full.index(f) for f in check_features]
    train_end=int(origins[0])-1;tr=np.arange(train_end-window+1,train_end+1)
    check_x=np.stack([matrices[f] if f not in WEEKDAY else np.broadcast_to(((dates+pd.Timedelta(days=1)).dayofweek==int(f.rsplit('_',1)[1])-1).astype(float),(n,len(dates))) for f in check_features],axis=2)
    direct_b,direct_a,*_=fit_fe(check_x[:,tr],counts[:,tr+1])
    direct=np.einsum('ntp,p->nt',check_x[:,origins[:length]],direct_b)+direct_a[:,None]
    assert np.allclose(predict(check_features)[0,:,:length],direct,atol=1e-7,rtol=1e-7)

    units={f:[f] for f in numeric};units['target_month']=GROUPS['month']
    protocol={'source':str(source.resolve()),'source_sha256':hashlib.sha256(source.read_bytes()).hexdigest(),
        'regions':n,'window':window,'horizons':[1,2,3],'retrain_every_days':7,
        'selection_period':'2021-01-01 to 2021-06-30, all targets inside period',
        'later_validation_period':'2021-07-01 to 2021-12-31, features frozen before scoring',
        'minimum_relative_MAE_improvement':min_relative_improvement,
        'methods':{'BACKWARD':'Start FULL; accept best single deletion; refit/repeat.',
            'FORWARD':'Start with weekday and ZIP FE only; accept best addition; refit/repeat.',
            'STEPWISE':'Start with weekday and ZIP FE only; best addition then repeatedly recheck best deletion until none accepted; repeat additions.'},
        'candidate_units':units,'forced_columns':WEEKDAY,'region_FE':True,'full_columns':full,
        'stopping_rule':'Accept only relative mean selection MAE reduction >=0.1%; deterministic feature-name tie break. No p-value criterion.',
        'scope':'Same fixed regions, forecast origins, horizons, training windows and inputs for every method.',
        'inference':'Exploratory. Adaptive search and repeatedly examined 2021 data preclude confirmatory or causal p-value claims. Holm over inspected tests does not fix adaptive selection.',
        'test_2022_2023_evaluated':False}
    (folder/'protocol.json').write_text(json.dumps(protocol,indent=2),encoding='utf-8')
    pd.DataFrame({'zip':ids}).to_csv(folder/'common_regions.csv',index=False)
    pd.DataFrame(timing).to_csv(folder/'training_timing.csv',index=False)
    unit_order=list(units)
    def columns(chosen):
        include=set(sum([units[u] for u in chosen],[])+WEEKDAY)
        return [f for f in full if f in include]
    # Cache only date-level selection losses, never later-period evaluation losses.
    memo={};predict_calls=0;steps=[];trial_rows=[]
    def score(chosen):
        nonlocal predict_calls
        key=tuple(u for u in unit_order if u in chosen)
        if key not in memo:
            pred=predict(columns(key));predict_calls+=1
            ae,mse,bias=losses(pred,selection_mask)
            memo[key]={'MAE':float(ae.mean()),'RMSE':float(np.sqrt(mse.mean(axis=1)).mean()),
                'daily_ae':ae,'daily_mse':mse,'bias':bias}
        return memo[key]
    draws=block_indices(int(selection_mask.sum()),14,bootstrap_repetitions,20261030)
    def attempt(method,phase,chosen,labels,step):
        cur=score(chosen);round_rows=[];best=None
        for label in sorted(labels):
            new=set(chosen)
            if phase=='add':new.add(label)
            else:new.remove(label)
            candidate=score(new)
            d=(cur['daily_ae']-candidate['daily_ae']).mean(axis=0)
            delta,se,pv=hac_mean_test(d,14)
            low,high=np.quantile(d[draws].mean(axis=1),[.025,.975])
            row={'method':method,'step':step,'action':phase,'candidate_unit':label,
                'current_MAE':cur['MAE'],'candidate_MAE':candidate['MAE'],'current_RMSE':cur['RMSE'],'candidate_RMSE':candidate['RMSE'],
                'delta_MAE_current_minus_candidate':delta,'relative_MAE_improvement':delta/cur['MAE'],
                'ci95_low_pointwise':low,'ci95_high_pointwise':high,'p_HAC_raw_exploratory':pv,
                'candidate_columns':len(columns(new)),'chosen':False,'candidate_features':';'.join(columns(new))}
            for h in range(3):row[f'candidate_MAE_h{h+1}']=float(candidate['daily_ae'][h].mean())
            round_rows.append(row)
            if best is None or (candidate['MAE'],label)<(best[0],best[1]):best=(candidate['MAE'],label,new,row)
        accepted=best is not None and (cur['MAE']-best[0])/cur['MAE']>=min_relative_improvement
        if accepted:best[3]['chosen']=True
        trial_rows.extend(round_rows)
        steps.append({'method':method,'step':step,'action':phase,'accepted':accepted,
            'unit':best[1] if best is not None else 'NONE','MAE_before':cur['MAE'],
            'MAE_after':best[0] if accepted else cur['MAE'],
            'columns_after':len(columns(best[2] if accepted else chosen)),
            'features_after':';'.join(columns(best[2] if accepted else chosen))})
        pd.DataFrame(steps).to_csv(folder/'selection_paths.csv',index=False)
        pd.DataFrame(trial_rows).to_csv(folder/'candidate_trials.csv',index=False)
        print(f'{method} step {step} {phase}: {best[1] if best else "NONE"}, MAE {cur["MAE"]:.6f} -> {best[0] if best else cur["MAE"]:.6f}, accepted={accepted}',flush=True)
        return (best[2] if accepted else chosen),accepted
    selected={}
    chosen=set(unit_order);step=0
    while chosen:
        step+=1;chosen,accepted=attempt('BACKWARD','drop',chosen,chosen,step)
        if not accepted:break
    selected['BACKWARD']=chosen.copy()
    chosen=set();step=0
    while set(unit_order)-chosen:
        step+=1;chosen,accepted=attempt('FORWARD','add',chosen,set(unit_order)-chosen,step)
        if not accepted:break
    selected['FORWARD']=chosen.copy()
    chosen=set();step=0
    while set(unit_order)-chosen:
        step+=1;chosen,accepted=attempt('STEPWISE','add',chosen,set(unit_order)-chosen,step)
        if not accepted:break
        while chosen:
            step+=1;chosen,deleted=attempt('STEPWISE','drop',chosen,chosen,step)
            if not deleted:break
        assert step<200,'Unexpected nontermination'
    selected['STEPWISE']=chosen.copy()
    # Persist all final lists before examining later-period losses.
    selection={method:{'candidate_units':sorted(chosen),'selected_columns':columns(chosen),
        'removed_columns':[f for f in full if f not in columns(chosen)],'feature_columns':len(columns(chosen)),
        'forced_weekday_columns':6,'region_FE_additional':True} for method,chosen in selected.items()}
    (folder/'selected_features_by_method.json').write_text(json.dumps(selection,indent=2),encoding='utf-8')
    inclusion=[]
    for feature in full:
        row={'feature':feature,'forced':feature in WEEKDAY}
        for method in selected:row[method]=feature in selection[method]['selected_columns']
        inclusion.append(row)
    pd.DataFrame(inclusion).to_csv(folder/'feature_inclusion_comparison.csv',index=False)
    trials=pd.DataFrame(trial_rows)
    trials['p_Holm_all_search_trials_exploratory']=holm_adjust(trials.p_HAC_raw_exploratory)
    trials.to_csv(folder/'candidate_trials.csv',index=False)
    models=['FULL','BACKWARD','FORWARD','STEPWISE'];predictions=[];rank_logs=[]
    for method in models:
        features=full if method=='FULL' else selection[method]['selected_columns']
        pred,rank_log=predict(features,True);predictions.append(pred)
        for row in rank_log:row['method']=method
        rank_logs.extend(rank_log)
    results=[];daily=[];comparisons=[]
    for period,mask in [('selection',selection_mask),('later_validation',assessment_mask)]:
        scores={}
        for method,pred in zip(models,predictions):
            ae,mse,bias=losses(pred,mask);scores[method]=(ae,mse)
            for h in range(3):
                results.append({'period':period,'method':method,'horizon':h+1,'feature_columns':len(full) if method=='FULL' else selection[method]['feature_columns'],
                    'regions':n,'forecast_dates':int(mask.sum()),'MAE':float(ae[h].mean()),'RMSE':float(np.sqrt(mse[h].mean())),'bias':float(bias[h])})
                for date,a,m in zip(dates[origins[mask]],ae[h],mse[h]):daily.append({'period':period,'method':method,'horizon':h+1,'date':date,'MAE':a,'MSE':m})
        indices=block_indices(int(mask.sum()),14,bootstrap_repetitions,20261031)
        for ia,baseline in enumerate(models):
            for candidate in models[ia+1:]:
                d=(scores[baseline][0]-scores[candidate][0]).mean(axis=0)
                delta,se,pv=hac_mean_test(d,14)
                low,high=np.quantile(d[indices].mean(axis=1),[.025,.975])
                comparisons.append({'period':period,'baseline':baseline,'candidate':candidate,'delta_MAE_baseline_minus_candidate':delta,
                    'ci95_low_pointwise':low,'ci95_high_pointwise':high,'p_HAC_raw_exploratory':pv})
    results=pd.DataFrame(results);comparison=pd.DataFrame(comparisons)
    comparison['p_Holm_all_12_exploratory']=holm_adjust(comparison.p_HAC_raw_exploratory)
    summary=results.groupby(['period','method','feature_columns'],as_index=False)[['MAE','RMSE','bias']].mean()
    summary.to_csv(folder/'method_comparison_summary.csv',index=False)
    results.to_csv(folder/'method_comparison_by_horizon.csv',index=False)
    comparison.to_csv(folder/'paired_method_comparisons.csv',index=False)
    pd.DataFrame(daily).to_csv(folder/'final_daily_losses.csv',index=False)
    pd.DataFrame(rank_logs).to_csv(folder/'rank_logs.csv',index=False)
    np.savez_compressed(folder/'final_predictions.npz',raw_predictions=np.stack(predictions),actual_cases=actual,models=np.array(models),zip=ids,date=dates[origins].to_numpy(),horizons=np.array([1,2,3]),selection_mask=selection_mask,assessment_mask=assessment_mask)
    report='# Backward, Forward and Stepwise feature selection\n\n'
    report+=f'Common {n} regions; 30-day windows, weekly refits, horizons 1/2/3. All methods use the same candidate pool, forced weekday controls and ZIP FE. A numeric variable is one selection unit; month encoding is one categorical unit. Initial FULL: {len(full)} columns.\n\n'
    report+='## Shared rule\n\nChoose the candidate with the lowest January-June 2021 mean MAE across horizons. Accept an addition/deletion only if relative MAE improves at least 0.1% over the current specification. Alphabetical tie break. Refit after each accepted change. RMSE is recorded, not optimized. This is a heuristic predictive selection rule, not a coefficient p-value stepwise procedure.\n\n'
    report+='BACKWARD starts FULL and removes. FORWARD starts with weekday/ZIP controls and adds. STEPWISE starts with controls, adds, then repeatedly checks deletions before trying another addition.\n\n'
    report+='## Mean metrics\n\n| Period | Method | Columns | MAE | RMSE |\n|---|---|---:|---:|---:|\n'
    for row in summary.itertuples():report+=f'| {row.period} | {row.method} | {row.feature_columns} | {row.MAE:.6f} | {row.RMSE:.6f} |\n'
    report+='\nLists are frozen before July-December validation scoring. Later outcomes may enter subsequent rolling parameter refits; membership never changes. Earlier work already examined 2021, so this is exploratory temporal assessment, not an untouched final test. No 2022-2023 test scores are generated.\n\n'
    report+='All search p-values are exploratory: HAC on date-level paired losses, Holm across examined search trials. Adaptive selection invalidates a confirmatory interpretation; correction does not repair search bias. Method comparisons adjust 12 pairwise tests across both periods. Pointwise 14-day bootstrap intervals resample fixed forecast errors, not the entire selection pipeline. Do not choose a definitive final method from later-validation ranking without acknowledging this additional selection.\n\n'
    report+='Start with method_comparison_summary.csv, feature_inclusion_comparison.csv and selection_paths.csv. selected_features_by_method.json contains exact lists. Rank logs distinguish inactive training calendar/constant columns from feature removal. Static population/density/commuting summaries remain tree-model candidates.\n'
    (folder/'RESULTS.md').write_text(report,encoding='utf-8')
    verification={'self_checks_passed':True,'cached_subset_OLS_matches_direct_OLS':True,'regions':n,'forecast_dates':v,
        'unique_search_specifications':len(memo),'search_trial_records':len(trials),'all_predictions_finite':True,
        'lists_frozen_before_later_losses':True,'test_evaluated':False}
    (folder/'verification.json').write_text(json.dumps(verification,indent=2),encoding='utf-8')
    print(summary.to_string(index=False),flush=True);print('Saved to:',folder.resolve(),flush=True)
    return pd.DataFrame(steps),summary,selection,folder


In [4]:
SOURCE = Path.cwd() / 'nj_daily_feature_outputs' / 'nj_daily_features.csv.gz'
OUTPUT = Path.cwd() / 'individual_feature_selection_outputs'
paths, comparison, selected, RUN_FOLDER = run_three_method_selection(SOURCE, OUTPUT)
display(comparison)
display(paths)
print(RUN_FOLDER.resolve())


BACKWARD step 1 drop: cases_mean_14d, MAE 1.066634 -> 1.064061, accepted=True
BACKWARD step 2 drop: days_since_last_observed_case_censored, MAE 1.064061 -> 1.061704, accepted=True
BACKWARD step 3 drop: target_month, MAE 1.061704 -> 1.059605, accepted=True
BACKWARD step 4 drop: case_rate_14d_per_100k, MAE 1.059605 -> 1.059074, accepted=False
FORWARD step 1 add: cases_mean_7d, MAE 1.283056 -> 1.098877, accepted=True
FORWARD step 2 add: nj_covid_pressure_7d, MAE 1.098877 -> 1.061669, accepted=True
FORWARD step 3 add: cases_today, MAE 1.061669 -> 1.060438, accepted=True
FORWARD step 4 add: case_rate_7d_per_100k, MAE 1.060438 -> 1.059429, accepted=False
STEPWISE step 1 add: cases_mean_7d, MAE 1.283056 -> 1.098877, accepted=True
STEPWISE step 2 drop: cases_mean_7d, MAE 1.098877 -> 1.283056, accepted=False
STEPWISE step 3 add: nj_covid_pressure_7d, MAE 1.098877 -> 1.061669, accepted=True
STEPWISE step 4 drop: nj_covid_pressure_7d, MAE 1.061669 -> 1.098877, accepted=False
STEPWISE step 5 add: 

,period,method,feature_columns,MAE,RMSE,bias
0,later_validation,BACKWARD,21,1.178472,2.018709,-0.011035
1,later_validation,FORWARD,9,1.171625,2.013175,-0.038075
2,later_validation,FULL,34,1.180819,2.023986,0.021619
3,later_validation,STEPWISE,9,1.171625,2.013175,-0.038075
4,selection,BACKWARD,21,1.059605,1.808737,0.098616
5,selection,FORWARD,9,1.060438,1.810525,0.120204
6,selection,FULL,34,1.066634,1.812614,0.117538
7,selection,STEPWISE,9,1.060438,1.810525,0.120204


,method,step,action,accepted,unit,MAE_before,MAE_after,columns_after,features_after
0,BACKWARD,1,drop,True,cases_mean_14d,1.066634,1.064061,33,cases_today;cases_lag_1;cases_lag_7;cases_lag_...
1,BACKWARD,2,drop,True,days_since_last_observed_case_censored,1.064061,1.061704,32,cases_today;cases_lag_1;cases_lag_7;cases_lag_...
2,BACKWARD,3,drop,True,target_month,1.061704,1.059605,21,cases_today;cases_lag_1;cases_lag_7;cases_lag_...
3,BACKWARD,4,drop,False,case_rate_14d_per_100k,1.059605,1.059605,21,cases_today;cases_lag_1;cases_lag_7;cases_lag_...
4,FORWARD,1,add,True,cases_mean_7d,1.283056,1.098877,7,cases_mean_7d;target_weekday_2;target_weekday_...
5,FORWARD,2,add,True,nj_covid_pressure_7d,1.098877,1.061669,8,cases_mean_7d;nj_covid_pressure_7d;target_week...
6,FORWARD,3,add,True,cases_today,1.061669,1.060438,9,cases_today;cases_mean_7d;nj_covid_pressure_7d...
7,FORWARD,4,add,False,case_rate_7d_per_100k,1.060438,1.060438,9,cases_today;cases_mean_7d;nj_covid_pressure_7d...
8,STEPWISE,1,add,True,cases_mean_7d,1.283056,1.098877,7,cases_mean_7d;target_weekday_2;target_weekday_...
9,STEPWISE,2,drop,False,cases_mean_7d,1.098877,1.098877,7,cases_mean_7d;target_weekday_2;target_weekday_...


C:\Users\41240\OneDrive\文档\GitHub\practicum-project\Weikun\individual_feature_selection_outputs\three_methods_20261026T153141838619Z


## Read results

1. method_comparison_summary.csv: selection/later-validation mean MAE, RMSE, bias and column counts.
2. feature_inclusion_comparison.csv: retained features side by side.
3. selection_paths.csv: order of accepted changes and stopping attempts.
4. method_comparison_by_horizon.csv: compare days 1/2/3 separately.
5. paired_method_comparisons.csv: differences, pointwise block CIs, exploratory HAC and Holm p-values.
6. candidate_trials.csv: all considered additions/deletions, not just winners.
7. selected_features_by_method.json: exact frozen lists.

Adaptive search and repeatedly used validation make p-values exploratory. Holm adjustment does not account fully for selection. Bootstrap intervals resample fixed forecast errors, not the entire model-selection pipeline. Check 60-day window stability and outbreak/spatial outputs separately before finalizing.


## Optional report figures

Requires matplotlib in the notebook kernel. This creates PNG (300 dpi) and SVG copies inside the run folder.


In [5]:
from pathlib import Path
import os

def plot_method_comparison(folder):
    folder=Path(folder);dest=folder/'figures';dest.mkdir(exist_ok=True)
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    import pandas as pd
    import numpy as np
    order=['FULL','BACKWARD','FORWARD','STEPWISE']
    s=pd.read_csv(folder/'method_comparison_summary.csv')
    later=s.loc[s.period.eq('later_validation')].set_index('method').reindex(order)
    colors=['#94a3b8','#2563eb','#059669','#7c3aed']
    plt.rcParams.update({'font.size':11,'axes.spines.top':False,'axes.spines.right':False})
    fig,axes=plt.subplots(1,2,figsize=(12,4.8))
    for ax,metric in zip(axes,['MAE','RMSE']):
        bars=ax.bar(order,later[metric],color=colors)
        ax.bar_label(bars,fmt='%.4f',padding=4)
        ax.set_title(f'Later-validation {metric}');ax.set_ylabel('Recorded-onset cases')
        ax.set_ylim(0,later[metric].max()*1.18);ax.grid(axis='y',alpha=.15);ax.set_axisbelow(True)
    fig.suptitle('Backward, Forward and Stepwise: same NJ forecast task')
    fig.text(.5,.015,'585 regions | 30-day training | Weekly refits | Horizons 1/2/3 | Features selected in early 2021, frozen for later 2021',ha='center',fontsize=9)
    fig.tight_layout(rect=[0,.06,1,.95])
    for extension in ['png','svg']:fig.savefig(dest/f'01_method_comparison.{extension}',dpi=300,bbox_inches='tight')
    plt.close(fig)
    inclusion=pd.read_csv(folder/'feature_inclusion_comparison.csv')
    inclusion=inclusion.loc[~inclusion.forced].copy()
    # Present all month dummies as the single categorical selection unit.
    month=inclusion.feature.str.startswith('target_month_')
    if month.any():
        row={'feature':'target_month (categorical block)'}
        for m in order[1:]:row[m]=bool(inclusion.loc[month,m].any())
        inclusion=pd.concat([inclusion.loc[~month],pd.DataFrame([row])],ignore_index=True)
    data=inclusion[order[1:]].to_numpy(dtype=int)
    fig,ax=plt.subplots(figsize=(8,8))
    ax.imshow(data,cmap=matplotlib.colors.ListedColormap(['#f1f5f9','#2563eb']),vmin=0,vmax=1,aspect='auto')
    ax.set_xticks(range(3),order[1:]);ax.set_yticks(range(len(inclusion)),inclusion.feature)
    for i in range(data.shape[0]):
        for j in range(3):ax.text(j,i,'Keep' if data[i,j] else 'Drop',ha='center',va='center',color='white' if data[i,j] else '#64748b',fontsize=9)
    ax.set_title('Feature membership after selection')
    fig.text(.5,.015,'Weekday controls and ZIP fixed effects retained in every method. Membership does not imply statistical significance.',ha='center',fontsize=8)
    fig.tight_layout(rect=[0,.04,1,1])
    for extension in ['png','svg']:fig.savefig(dest/f'02_feature_membership.{extension}',dpi=300,bbox_inches='tight')
    plt.close(fig)
    return dest


In [6]:
try:
    print(plot_method_comparison(RUN_FOLDER))
except ImportError:
    print('Optional figures require matplotlib: %pip install matplotlib')


C:\Users\41240\OneDrive\文档\GitHub\practicum-project\Weikun\individual_feature_selection_outputs\three_methods_20261026T153141838619Z\figures


## Reopen completed results without rerunning selection

The following cell can be run independently to read the already completed comparison.


In [7]:
from pathlib import Path
import pandas as pd
EXISTING = Path.cwd() / 'individual_feature_selection_outputs' / 'three_methods_20261026T152516441253Z'
display(pd.read_csv(EXISTING / 'method_comparison_summary.csv'))
display(pd.read_csv(EXISTING / 'feature_inclusion_comparison.csv'))
print((EXISTING / 'RESULTS.md').read_text(encoding='utf-8'))


,period,method,feature_columns,MAE,RMSE,bias
0,later_validation,BACKWARD,21,1.178472,2.018709,-0.011035
1,later_validation,FORWARD,9,1.171625,2.013175,-0.038075
2,later_validation,FULL,34,1.180819,2.023986,0.021619
3,later_validation,STEPWISE,9,1.171625,2.013175,-0.038075
4,selection,BACKWARD,21,1.059605,1.808737,0.098616
5,selection,FORWARD,9,1.060438,1.810525,0.120204
6,selection,FULL,34,1.066634,1.812614,0.117538
7,selection,STEPWISE,9,1.060438,1.810525,0.120204


,feature,forced,BACKWARD,FORWARD,STEPWISE
0,cases_today,False,True,True,True
1,cases_lag_1,False,True,False,False
2,cases_lag_7,False,True,False,False
3,cases_lag_14,False,True,False,False
4,cases_mean_7d,False,True,True,True
5,cases_mean_14d,False,False,False,False
6,case_std_7d,False,True,False,False
7,case_std_14d,False,True,False,False
8,case_acceleration,False,True,False,False
9,case_jerk,False,True,False,False


# Backward, Forward and Stepwise feature selection

Common 585 regions; 30-day windows, weekly refits, horizons 1/2/3. All methods use the same candidate pool, forced weekday controls and ZIP FE. A numeric variable is one selection unit; month encoding is one categorical unit. Initial FULL: 34 columns.

## Shared rule

Choose the candidate with the lowest January-June 2021 mean MAE across horizons. Accept an addition/deletion only if relative MAE improves at least 0.1% over the current specification. Alphabetical tie break. Refit after each accepted change. RMSE is recorded, not optimized. This is a heuristic predictive selection rule, not a coefficient p-value stepwise procedure.

BACKWARD starts FULL and removes. FORWARD starts with weekday/ZIP controls and adds. STEPWISE starts with controls, adds, then repeatedly checks deletions before trying another addition.

## Mean metrics

| Period | Method | Columns | MAE | RMSE |
|---|---|---:|---:|---:|
| later_validation | BACKWARD | 21 |